# Hold maps from video stills

Builds the hold maps from stills taken from route videos.

1. **Before labeling**: for every camera pos take the sharpest empty-wall frame from one of its videos and save it
   - `hold-maps/stills/posX.png`: these are what should be manually labeled in roboflow.
   - `hold-maps/posX_frame.png`: same frame for ref.
2. **After labeling:** downloads roboflow dataset export and write
   - `hold-maps/holds.csv`: one row per hold (position, hold ID, class, color, center, bounding box, area)
   - `hold-maps/hold_polygons.json`: each hold's outline, for contact extraction and cropping
   - `hold-maps/posX_numbered.png`: each frame with every hold outlined and numbered
   - `hold-maps/checks/posX_overlay.jpg`: the outlines drawn on the reference frame, to check them
   - `hold-maps/routes.csv`: each route's holds, pre-filled by color, to finish by hand

## Cell 1: Setup

In [ ]:
!pip install -q roboflow
import os, re, json, glob
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict, Counter
from google.colab import drive, userdata

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ECS-174-Project'
VIDEO_DIRS = [f'{BASE}/original-footage', f'{BASE}/more-footage']
MAP_DIR = f'{BASE}/hold-maps'
STILLS_DIR = f'{MAP_DIR}/stills'
META_CSV = f'{BASE}/metadata.csv'
VIDEO_COL, POS_COL, COLOR_COL = 'Filename', 'Position_ID', 'Route Grade/Color'
START_COL = 'Climb_Start'                   # seconds; frames before this are the empty wall
ROBOFLOW_PROJECT, ROBOFLOW_VERSION = 'bouldering-holds-stage-2', 3

# Positions left out entirely (camera too shaky to align)
EXCLUDE_POSITIONS = {'pos2', 'pos4', 'pos6', 'pos7', 'pos11', 'pos18', 'pos22', 'pos25'}
VIDEO_EXTS = ('.mp4')

os.makedirs(f'{MAP_DIR}/checks', exist_ok=True)
os.makedirs(STILLS_DIR, exist_ok=True)

def normalize_pos(s):
    """'Pos02', 'pos2', 'pos02_still' -> 'pos2'."""
    m = re.search(r'pos0*(\d+)', str(s), re.IGNORECASE)
    return f'pos{int(m.group(1))}' if m else None

## Cell 2: Metadata, route IDs and videos

- Build route IDs from filenames: `izzy-green-1` and `izzy-green-1.2` (attempt 2) both become `green1`.
- Drop positions in `EXCLUDE_POSITIONS`, with their videos and routes.
- Find each video file in the footage folders

In [ ]:
meta = pd.read_csv(META_CSV)
meta = meta[meta[VIDEO_COL].notna()].copy()
meta['pos'] = meta[POS_COL].map(normalize_pos)

def strip_video_ext(name):
    name = str(name).strip()
    for ext in VIDEO_EXTS:
        if name.lower().endswith(ext):
            return name[:-len(ext)]
    return name

def route_id_of(row):
    parts = strip_video_ext(row[VIDEO_COL]).lower().split('-')
    color_word, number = parts[-2], parts[-1].split('.')[0]
    return f"{color_word}{number}"

dropped = meta[meta['pos'].isin(EXCLUDE_POSITIONS)]
meta = meta[~meta['pos'].isin(EXCLUDE_POSITIONS) & meta['pos'].notna()].copy()
meta['route_id'] = meta.apply(route_id_of, axis=1)
meta['route_color'] = meta[COLOR_COL].astype(str).str.lower()
print(f'Excluded positions remove {len(dropped)} videos; {len(meta)} videos remain at {meta.pos.nunique()} positions')

video_files = {}
for d in VIDEO_DIRS:
    for root, _, files in os.walk(d):
        for f in files:
            if f.lower().endswith(VIDEO_EXTS):
                video_files.setdefault(strip_video_ext(f).lower(), os.path.join(root, f))

def find_video(name):
    return video_files.get(strip_video_ext(name).lower())

missing = [v for v in meta[VIDEO_COL] if find_video(v) is None]
print(f'{len(missing)} videos not found:', missing)
climb_start = {strip_video_ext(r[VIDEO_COL]).lower(): r[START_COL] for _, r in meta.iterrows()}

## Cell 3: Take a still from each position's video

For each position, reads every 5th frame of the empty-wall part of one of its video and keep the sharpest one.

In [ ]:
def empty_wall_frames(video, seconds, step=5):
    """Every `step`-th frame from the first `seconds` of the video."""
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    out = []
    for i in range(max(1, int(seconds * fps))):
        ok, f = cap.read()
        if not ok:
            break
        if i % step == 0:
            out.append((i, f))
    cap.release()
    return out

def sharpness(frame):
    # sharpess check
    return cv2.Laplacian(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()

stills = []
for pos, g in meta.groupby('pos'):
    still_path = f'{STILLS_DIR}/{pos}.png'
    if os.path.exists(still_path):
        stills.append({'position': pos, 'status': 'already exists (kept)'})
        continue
    row = None
    for v in g[VIDEO_COL]:
        video = find_video(v)
        if video is None:
            continue
        start = climb_start.get(strip_video_ext(v).lower())
        window = max(1.0, float(start) - 0.5) if pd.notna(start) else 3.0    # stop before the climber steps in
        cands = empty_wall_frames(video, window)
        if cands:
            idx, frame = max(cands, key=lambda c: sharpness(c[1]))
            cv2.imwrite(still_path, frame)
            cv2.imwrite(f'{MAP_DIR}/{pos}_frame.png', frame)
            row = {'position': pos, 'status': 'saved', 'video': os.path.basename(video), 'frame': idx,
                   'size': f'{frame.shape[1]}x{frame.shape[0]}', 'sharpness': round(sharpness(frame), 1)}
            break
    stills.append(row or {'position': pos, 'status': 'NO READABLE VIDEO'})
display(pd.DataFrame(stills).sort_values('position'))

## Cell 4: Download the dataset after labeling is done

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace("bouldering-hold-labeling").project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("coco-segmentation")
print(dataset.location)

## Cell 5: Load the labels and match them to positions

Matches each labeled image to its position by filename.

In [ ]:
images, anns_by_img, cats = {}, defaultdict(list), {}
for coco_file in glob.glob(f'{dataset.location}/*/_annotations.coco.json'):
    folder = os.path.dirname(coco_file)
    coco = json.load(open(coco_file))
    for c in coco['categories']:
        cats[c['id']] = c['name']
    for im in coco['images']:
        key = (coco_file, im['id'])            # image ids restart in each split's file
        images[key] = {**im, 'path': os.path.join(folder, im['file_name'])}
    for a in coco['annotations']:
        anns_by_img[(coco_file, a['image_id'])].append(a)

def position_of(file_name):
    """Roboflow renames 'pos3.png' so recover 'pos3'."""
    return normalize_pos(file_name.split('.rf.')[0])

frames, seen = {}, {}
for key, im in list(images.items()):
    pos = position_of(im['file_name'])
    im['pos'] = pos
    if pos is None or pos in EXCLUDE_POSITIONS:
        images.pop(key)
        continue
    seen[pos] = im['file_name']
    frames[pos] = cv2.imread(f'{MAP_DIR}/{pos}_frame.png')     # reference frame

counts = Counter(cats[a['category_id']] for key in images for a in anns_by_img[key])
print(f'{len(images)} positions, {sum(counts.values())} labeled objects')
print('Objects per class:', dict(sorted(counts.items())))

## Cell 6: Build the hold table

Read both label formats: polygons and RLE masks traced into outlines. Hold IDs go bottom to top, then left to right. Save `holds.csv` and `hold_polygons.json`.

In [ ]:
!pip install -q pycocotools
from pycocotools import mask as mask_utils

def seg_to_mask(seg):
    """COCO RLE mask -> binary image."""
    h, w = seg['size']
    rle = dict(seg)
    if isinstance(rle['counts'], list):                          # uncompressed RLE
        rle = mask_utils.frPyObjects(rle, h, w)
    elif isinstance(rle['counts'], str):                         # compressed RLE as text
        rle['counts'] = rle['counts'].encode('ascii')
    return np.ascontiguousarray(mask_utils.decode(rle), dtype=np.uint8)

def polygon_parts(seg):
    """Outline(s) of one label as point arrays: polygons as they are, RLE masks traced into outlines."""
    if isinstance(seg, dict):
        contours, _ = cv2.findContours(seg_to_mask(seg), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return [c.reshape(-1, 2).astype(np.float32) for c in contours
                if len(c) >= 3 and cv2.contourArea(c) >= 4]       # drop tiny specks
    return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg if len(p) >= 6]

def parse_class(name):
    """'red-hold' or 'hold-red' -> ('hold', 'red'); 'down' -> ('down', ''); 'volume' -> ('volume', '')."""
    n = name.lower()
    if n.endswith('-hold'):
        return 'hold', n[:-len('-hold')]
    if n.startswith('hold-'):
        return 'hold', n[len('hold-'):]
    return n, ''

hold_rows, polygons, items_by_pos = [], {}, {}
for key, im in images.items():
    pos = im['pos']
    items = []
    for a in anns_by_img[key]:
        parts = polygon_parts(a['segmentation'])
        if not parts:
            continue
        pts = np.concatenate(parts)
        x0, y0 = pts.min(axis=0)
        x1, y1 = pts.max(axis=0)
        areas = [cv2.contourArea(p) for p in parts]
        M = cv2.moments(parts[int(np.argmax(areas))])          # center of the largest part
        cx, cy = (M['m10'] / M['m00'], M['m01'] / M['m00']) if M['m00'] else ((x0 + x1) / 2, (y0 + y1) / 2)
        cls, color = parse_class(cats[a['category_id']])
        items.append({'cls': cls, 'color': color, 'cx': cx, 'cy': cy, 'parts': parts,
                      'bbox': (x0, y0, x1 - x0, y1 - y0), 'area': sum(areas)})

    items.sort(key=lambda d: (-d['cy'], d['cx']))              # bottom to top, then left to right
    for hold_id, d in enumerate(items, start=1):
        d['hold_id'] = hold_id
        bx, by, bw, bh = d['bbox']
        hold_rows.append({'position_id': pos, 'hold_id': hold_id, 'cls': d['cls'], 'color': d['color'],
                          'center_x': round(float(d['cx']), 1), 'center_y': round(float(d['cy']), 1),
                          'bbox_x': round(float(bx), 1), 'bbox_y': round(float(by), 1),
                          'width': round(float(bw), 1), 'height': round(float(bh), 1),
                          'area': round(float(d['area']), 1)})
        polygons.setdefault(pos, {})[str(hold_id)] = [p.round(1).tolist() for p in d['parts']]
    items_by_pos[pos] = items

holds_df = pd.DataFrame(hold_rows).sort_values(['position_id', 'hold_id'])
n_labels = sum(len(anns_by_img[k]) for k in images)
print(f'{n_labels} labels -> {len(holds_df)} holds in the table')
holds_df.to_csv(f'{MAP_DIR}/holds.csv', index=False)
json.dump(polygons, open(f'{MAP_DIR}/hold_polygons.json', 'w'))
print(holds_df.groupby('position_id').size().rename('holds').to_string())

## Cell 7: Draw the numbered images and overlay checks

Drawn on each position's reference frame. Yellow outlines = holds, magenta = volumes, blue = down-climb holds. `checks/posX_overlay.jpg` shows only the outlines, to confirm they sit on the holds.

In [ ]:
def draw_numbered(img, items, numbers=True):
    out = img.copy()
    s = max(img.shape[:2]) / 1500                              # scale lines and text to image size
    for d in items:
        color = {'hold': (0, 255, 255), 'volume': (255, 0, 255)}.get(d['cls'], (255, 160, 0))
        for p in d['parts']:
            cv2.polylines(out, [p.astype(np.int32)], True, color, max(1, int(2 * s)))
    if numbers:
        for d in items:
            txt, font, fs = str(d['hold_id']), cv2.FONT_HERSHEY_SIMPLEX, 0.7 * s
            (tw, th), _ = cv2.getTextSize(txt, font, fs, max(1, int(2 * s)))
            org = (int(d['cx'] - tw / 2), int(d['cy'] + th / 2))   # center the number on the hold
            cv2.putText(out, txt, org, font, fs, (0, 0, 0), max(3, int(5 * s)), cv2.LINE_AA)       # dark outline
            cv2.putText(out, txt, org, font, fs, (255, 255, 255), max(1, int(2 * s)), cv2.LINE_AA)  # white text
    return out

for pos, items in items_by_pos.items():
    cv2.imwrite(f'{MAP_DIR}/{pos}_numbered.png', draw_numbered(frames[pos], items))
    cv2.imwrite(f'{MAP_DIR}/checks/{pos}_overlay.jpg', draw_numbered(frames[pos], items, numbers=False))
print('Saved numbered images and overlays to', MAP_DIR)

## Cell 8: Pre-fill `routes.csv`

Add for each route at a labeled position, every hold of the route's color (`on_route=1`, role `middle`) and every volume (left blank to decide by hand). We finish this by ahdn by marking start and end holds, and fixing holds of the route's color that belong to another route in the same img.

In [ ]:
ROUTES_CSV = f'{MAP_DIR}/routes.csv'
COLORS = sorted(c for c in holds_df.loc[holds_df.cls == 'hold', 'color'].unique() if c)

existing = pd.read_csv(ROUTES_CSV) if os.path.exists(ROUTES_CSV) else pd.DataFrame(
    columns=['route_id', 'position_id', 'hold_id', 'cls', 'color', 'on_route', 'role'])
done = set(zip(existing.route_id.astype(str), existing.position_id.astype(str)))

new_rows = []
for route, pos, color_text in meta[['route_id', 'pos', 'route_color']].drop_duplicates().itertuples(index=False):
    if pos not in items_by_pos or (route, pos) in done:        # not labeled yet, or already filled in
        continue
    words = re.split(r'[^a-z]+', color_text)
    color = next((c for c in COLORS if c in words), None)      # match against the label colors
    if color is None:
        print(f'Could not match color "{color_text}" for route {route}; fill its holds in by hand.')
    for h in holds_df[holds_df.position_id == pos].itertuples():
        if h.cls == 'hold' and h.color == color:
            new_rows.append([route, pos, h.hold_id, h.cls, h.color, 1, 'middle'])
        elif h.cls == 'volume':
            new_rows.append([route, pos, h.hold_id, h.cls, '', '', ''])   # decide by hand

routes = pd.concat([existing, pd.DataFrame(new_rows, columns=existing.columns)], ignore_index=True)
routes.to_csv(ROUTES_CSV, index=False)
print(f'Added {len(new_rows)} rows for {len({(r[0], r[1]) for r in new_rows})} route/position pairs')